# Module 2: Online Retail Analytics — RFM Segmentation & Market Basket

Welcome to **Module 2**! In this lab, you will analyze 541,900 transaction records from a UK giftware wholesaler (`online_retail.xlsx`).

### Business Brief:
> *"Which customers and products actually make us money, and who is about to stop buying?"*

---

### Assignment Tasks Overview:
1. **Task 1**: Load `online_retail.xlsx` using Pandas, clean missing `CustomerID`s, and filter cancelled orders (Invoice starting with 'C').
2. **Task 2**: Calculate Total Sales (`Quantity * UnitPrice`) and filter invalid non-positive quantities.
3. **Task 3**: Monthly Revenue Analysis & Top Performing Countries.
4. **Task 4**: Customer RFM Segmentation (Calculate Recency, Frequency, Monetary scores).
5. **Task 5**: Market Basket Analysis — Discover product association rules for cross-selling recommendations.

In [2]:
# Task 1: Data Sanitization & Cleaning
import pandas as pd
import numpy as np

# Load dataset (pre-mounted in workspace)
df = pd.read_excel('online_retail.xlsx')

# Remove rows without Customer ID & cancelled orders
df_clean = df.dropna(subset=['CustomerID']).copy()
df_clean['InvoiceNo'] = df_clean['InvoiceNo'].astype(str)
df_clean = df_clean[~df_clean['InvoiceNo'].str.startswith('C')]

# Calculate Total Sales
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]
df_clean['TotalSales'] = df_clean['Quantity'] * df_clean['UnitPrice']

print(f"Original Rows: {len(df)} | Cleaned Rows: {len(df_clean)}")
df_clean.head(10)

### Task 2: RFM Customer Segmentation
Calculate Recency (days since last order), Frequency (order count), and Monetary (total spend) for each customer.

In [4]:
# Task 2: RFM Calculation
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalSales': 'sum'
}).rename(columns={'InvoiceDate': 'Recency', 'InvoiceNo': 'Frequency', 'TotalSales': 'Monetary'})

print("--- Top 10 Customers by Total Spend (Monetary) ---")
print(rfm.sort_values(by='Monetary', ascending=False).head(10))